# Customer Pilot Engineering

Turn an ambiguous customer request into an accepted, observable pilot with an operator handoff.

Run the lecture cells in sequence with Python 3.10 or later. The default exercises use bundled data and require no model account.


## Practice case · Meet the people who fix the building

Inez is embedded with a facilities contractor. Earlier demos suggested AI could route almost every maintenance issue. The shift lead shows her unclassified tickets, a legacy form, and a queue that must keep working during a network failure. They agree on a limited trial with manual ownership intact.

The workplace scenario is fictional.


## Discover and integrate


### Lecture 01 · Discovery and pilot acceptance

Translate a request into a workflow, baseline, and measurable acceptance criteria.

#### Translate vague enthusiasm into a measured goal

The sponsor asks for smart intake. The shift lead says reporters will still call if they cannot tell who owns a request.

Observe actual handoffs and name which delay the pilot must reduce. Establish manual median time, escalation behavior, and a safety gate. Ask the shift lead to confirm the expected human follow-up.

#### How it works

Observe who does the work, what they decide, how long it takes, and who can approve a change. Separate a business outcome from service behavior. Define acceptance by case and agree on a comparison before building.

##### Learn from the person doing the work

A pilot starts with a workflow and a customer decision. Watch how a real request moves through the current process, including the unofficial work people do to correct mistakes. Write the acceptance criterion in terms the shift lead can verify. Faster proposed routes are useful only if staff can recognize who owns the resulting ticket.

Use historical and synthetic cases to agree on what the pilot may touch. Establish the manual baseline, which work is sensitive, and when to ask a person. Decide which faults would pause the feature even if the average time improves. Protect an existing route for uncertain and restricted items.

Ask the customer to verify what an accepted result looks like using the evidence they normally work with. A successful demonstration is one repeatable observation, while acceptance also needs coverage and explicit ownership. Frame the smallest scope that can answer the customer’s real question.

#### Worked example

A facilities manager asks for faster intake. Observe manual triage and target lower median time while preserving all restricted routing decisions.

#### Try it

Use the baseline time measurements in the workbook. Write a target and a guardrail that would both be observable during a pilot.


In [ ]:
from statistics import median

baseline_minutes = (10, 11, 12, 13, 15)
median_target = median(baseline_minutes) * 0.75
assert median_target == 9
print("Median baseline and acceptance target", median(baseline_minutes), median_target)


**Compare your result**

A defensible starting contract targets a 25% reduction in median time and requires every restricted case to be handed off.


### Lecture 02 · Map a customer data contract

Normalize a source payload and publish a versioned integration boundary.

#### Keep source identity when you normalize fields

The upstream form sends an old category value and the same ticket can arrive twice. A malformed record should not silently enter a general queue.

Validate schema version and expected types at intake. Preserve source identity so staff can resolve errors. Route unknown categories to an explicit review state instead of assuming a default.

#### How it works

Map external fields to internal types and preserve provenance. Reject incomplete records before calling a downstream dependency. Agree who owns the upstream schema and define the service behavior if an unexpected version arrives.

##### Keep a record recognizable after normalization

An adapter maps between the customer's data and the application's contract. Validate the upstream schema version and required fields, then preserve identifiers and timestamps. People need these details to find the original ticket when a candidate route looks wrong or a record is delivered twice.

Handle three inputs separately. A known category maps to an allowed internal value. An unknown category becomes an explicit review item. An invalid value is rejected with a reason and, where permitted, a reference to the original record. Using a broad default for all three loses evidence about what happened.

Version the mapping and exercise a boundary case before deploying it. Ask the upstream owner how a schema or category changes are advertised. A customer can make a sensible update in their system and still break your assumptions. Give the operator a way to detect, pause, and recover from that mismatch.

#### Worked example

A partner sends a customer ticket ID, topic, and audience flag. Normalize known topics and route any unrecognized topic to review.

#### Try it

Run the adapter for the bundled records. Add a case for a field with the wrong type.


In [ ]:
from dataclasses import dataclass


@dataclass(frozen=True)
class Ticket:
    source_id: str
    topic: str
    restricted: bool


def normalize(payload):
    if payload.get("schema") != "facilities-v1":
        raise ValueError("unsupported schema")
    source = payload.get("ticket")
    topic = payload.get("queue")
    restricted = payload.get("restricted")
    if not isinstance(source, str) or not source.strip() or not isinstance(topic, str) or type(restricted) is not bool:
        raise ValueError("invalid ticket fields")
    return Ticket(source.strip(), topic.strip().lower(), restricted)


inputs = [
    {"schema": "facilities-v1", "ticket": "F1", "queue": " Heating ", "restricted": False},
    {"schema": "facilities-v1", "ticket": "F2", "queue": "heating", "restricted": True},
    {"schema": "facilities-v1", "ticket": "F3", "queue": "other", "restricted": False},
]
tickets = [normalize(row) for row in inputs]
assert tickets[0] == Ticket("F1", "heating", False)
try:
    normalize(inputs[0] | {"restricted": "false"})
except ValueError as error:
    print("Rejected malformed request", error)
else:
    raise AssertionError("wrong field type was accepted")


**Compare your result**

Treat the malformed input as a rejected record. Leave the original source identifier available for support.


## Evaluate and hand off


### Lecture 03 · Shadow acceptance and integration failures

Exercise normal, ambiguous, restricted, and unavailable-dependency cases before live use.

#### Shadow decisions while the old process still runs

During a rehearsal, the directory fails for a routine ticket and an unusual ticket arrives twice. The existing staff queue keeps moving.

Record candidate decisions without applying them. Compare against the staff route, retry recoverable failures, and test duplicate handling. This gives the customer useful acceptance evidence before changing the live process.

##### Follow one work order through the rehearsal

In the existing workflow, the desk retains ownership until the shift lead sees a filed ticket. In the rehearsal, the adapter records a proposed route without changing that ownership.

| Event | Candidate record | What the shift lead should see |
|---|---|---|
| Routine ticket, directory available | Propose the expected queue | A candidate destination beside the desk’s route |
| Directory unavailable | Request needs manual handling or retry | The original ticket remains visible to the desk |
| Same ticket arrives again | Return the prior result for that ticket ID | No second work order |

Ask an operator to perform these steps using the workbook output as evidence. They should be able to tell whether the pilot acted, whether staff still own the ticket, and when they need to do something. A green automated assertion can confirm a route; the rehearsal checks whether the customer can safely run the process.

#### How it works

A shadow run records proposed actions while the original workflow retains control. Test the same inputs for both workflows and compare decisions and timing. Make failures easy to retry without duplicating the outcome.

##### Compare decisions without changing operational ownership

In a shadow trial, the candidate produces proposed outputs while the original workflow retains control. Run both against the same ticket inputs and compare the observable result. A difference is a reason to inspect the case, not automatic proof that the candidate is wrong. The existing route can contain errors too.

Include a temporary directory failure, an uncertain category, and a repeated ticket. On failure, staff should still be able to find the original work. On replay, the candidate should refer to the existing record. When routing is uncertain, the system should describe the review state rather than claiming automatic success.

Ask the shift lead to walk through the evidence in order. Can they see who owns the ticket right now? Would the proposal improve triage? What would happen after disconnecting the service? These operational checks complement accuracy and help the customer decide when it is safe to introduce a live effect.

#### Worked example

When the routing dependency is unavailable, leave the ticket in manual triage. If an accepted request is delivered twice, return the existing route.

#### Try it

Compare the deterministic acceptance results. Add a replay for a successful request and a retry after dependency recovery.


In [ ]:
class Intake:
    def __init__(self):
        self.records = {}

    def shadow(self, ticket, lookup):
        saved = self.records.get(ticket.source_id)
        if saved:
            if saved[0] != ticket:
                raise ValueError("source ID reused with different inputs")
            return saved[1]
        try:
            route = lookup(ticket.topic) if not ticket.restricted else None
        except TimeoutError:
            return "retry_manual"
        result = route or "review"
        self.records[ticket.source_id] = (ticket, result)
        return result


def lookup(topic):
    return {"heating": "building-services"}.get(topic)


def unavailable(topic):
    raise TimeoutError("directory did not respond")


intake = Intake()
assert intake.shadow(tickets[0], unavailable) == "retry_manual"
assert "F1" not in intake.records
assert [intake.shadow(ticket, lookup) for ticket in tickets] == ["building-services", "review", "review"]
assert intake.shadow(tickets[0], unavailable) == "building-services"
assert len(intake.records) == 3


**Compare your result**

The successful replay returns the same result and creates one record. A transient failure leaves room for a successful retry.


### Lecture 04 · Rollout, operator handoff and stop rules

Write a release recommendation grounded in measured outcomes and assign an operational owner.

#### Hand over a decision someone can operate

The pilot reaches its time target on routine tickets. The shift lead worries about what happens when someone submits a restricted request tomorrow.

Show the restricted-case result and a working stop rule. Name the owner of manual fallback, provide a reproducible smoke case, and document how the customer can pause the pilot without the engineering team.

#### How it works

Report the cohort, case counts, observed limitations, and actions after launch. A stop rule links evidence to an action and owner. The handoff includes a smoke case, fallback route, data boundary, and how to read the service record.

##### The pilot is only as reversible as the handoff

A customer-facing release needs a named owner, a defined scope, a known smoke case, and an accessible fallback. Report which tasks passed and which still go to staff. Include the threshold that triggers pause and explain how to identify requests handled during the pilot.

Give the shift lead a small rehearsal. Have them recognize a failure, pause the feature, find an in-flight item, and confirm it remains in the manual queue. The ability to follow a runbook becomes part of acceptance evidence. A handoff is incomplete if preserving the workflow still requires the developer to be present.

Keep a log of responsibilities and a checkpoint for reassessment. A bounded approval can be extended later when the customer has enough evidence on new categories. A limited release that the operators can understand and reverse is often more valuable than a broader demonstration with uncertain ownership.

#### Worked example

The pilot meets its median time target, but a restricted case routes automatically. The owner keeps manual triage active and blocks expansion.

#### Try it

Review both bundled pilot result sets. Write a one-page handoff with the same triggers.


In [ ]:
def decide(observed_times, restricted_case_ok, completed_smoke_test):
    return (
        len(observed_times) >= len(baseline_minutes)
        and median(observed_times) <= median_target
        and restricted_case_ok
        and completed_smoke_test
    )


pilot_minutes = (6, 7, 8, 9, 11)
assert decide(pilot_minutes, True, True)
assert not decide(pilot_minutes, False, True)
assert not decide(pilot_minutes[:2], True, True)
print({
    "pilot_cohort": "simulated facilities intake",
    "cases": len(pilot_minutes),
    "median_minutes": median(pilot_minutes),
    "recommendation": "bounded pilot",
    "operator_owner": "facilities shift lead",
    "stop_rule": "route any restricted case manually and pause automation",
    "smoke_case": {"input_id": "F2", "expected": "review"},
})


**Compare your result**

Recommend controlled adoption only when the thresholds pass. Identify who can pause the service and how affected work is handled.


## Course project

Deliver a pilot for a simulated facilities team. Include the workflow baseline, an adapter, a replay-safe intake path, acceptance evidence, and a handoff.

Show an accepted request, unverifiable evidence, a temporary dependency failure, and a replay. Record the scope, pilot owner, stop rule, and what the customer needs to operate it.
